# 02 — Model Training: Recommendation + LLM Fine-Tuning

Trains both models using **Kubeflow Trainer SDK** with distributed DDP across multiple GPU nodes.

## Phase 1: Recommendation Model (Two-Tower)

| Setting | Value |
|---------|-------|
| Architecture | Two-Tower collaborative filtering |
| Framework | HuggingFace Trainer + DDP (NCCL) |
| Data | `s3://smartshop-features/interactions` |
| Output | `s3://smartshop-models/recommendation/best_model.pt` |

## Phase 2: LLM Fine-Tuning (LoRA + FSDP)

| Setting | Value |
|---------|-------|
| Base model | Mistral-7B-Instruct-v0.3 |
| Method | LoRA (r=16, α=32) + FSDP full_shard |
| Data | `s3://smartshop-features/llm_data` |
| Output | `s3://smartshop-models/llm-adapter` |

**Runs from:** RHOAI Workbench (in-cluster)  
**Runtime:** `torch-distributed` ClusterTrainingRuntime


In [4]:
%pip install -q kubeflow --no-cache-dir \
    --index-url https://console.redhat.com/api/pypi/public-rhai/rhoai/3.3/cuda12.9-ubi9/simple/
%pip install -q kubernetes boto3 tabulate pandas pyarrow feast redis pyyaml s3fs "fsspec[s3]" scikit-learn
%pip install -q yamlmagic --index-url https://pypi.org/simple
%load_ext yamlmagic


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Phase 1 Parameters — Recommendation Model

Edit these to customize Two-Tower training.


In [ ]:
%%yaml rec_params

# Data
data_dir: s3://smartshop-features
output_dir: s3://smartshop-models/recommendation
max_rows: 5000000                          # cap to avoid OOM on 32Gi nodes

# Hyperparameters
epochs: 10
batch_size: 2048
lr: 0.0003
embed_dim: 64
hidden_dim: 256

# Cluster topology
num_nodes: 2
gpus_per_node: 2
cpu_per_node: 8
memory_per_node: 32Gi


In [ ]:
from _config import *

RUNTIME = "torch-distributed"
TIMEOUT_SECONDS = 7200

validate()

## Initialize Kubeflow TrainerClient


In [6]:
import warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")

from kubernetes import client as k8s
from kubeflow.trainer import TrainerClient
from kubeflow.common.types import KubernetesBackendConfig

# In-cluster auth — workbench pod has a service account token
trainer = TrainerClient(
    KubernetesBackendConfig(namespace=NAMESPACE)
)

runtime = trainer.get_runtime(RUNTIME)
print(f"Runtime: {RUNTIME} ✓")

Runtime: torch-distributed ✓


---
# Phase 1: Recommendation Model Training

## 1.1 Rec Training Config


In [45]:
print(f"Data:       {rec_params['data_dir']}")
print(f"Output:     {rec_params['output_dir']}")
print(f"Topology:   {rec_params['num_nodes']} nodes x {rec_params['gpus_per_node']} GPUs = {rec_params['num_nodes'] * rec_params['gpus_per_node']} total GPUs")


Data:       s3://smartshop-features
Output:     s3://smartshop-models/recommendation
Topology:   4 nodes x 2 GPUs = 8 total GPUs


## 1.2 Define Training Function

Uses HuggingFace `transformers.Trainer` for automatic progress tracking, step-level logging,
checkpointing, and early stopping. The Kubeflow SDK serializes this into the TrainJob pod.
`torchrun` + `PET_*` env vars are injected automatically by the controller.


In [ ]:
def train_fn(
    data_dir: str,
    output_dir: str,
    epochs: int = 10,
    batch_size: int = 2048,
    lr: float = 3e-4,
    embed_dim: int = 64,
    hidden_dim: int = 256,
    max_rows: int = 5000000,
):
    import os, io, gc, time as _time
    import pandas as pd
    import numpy as np
    import torch
    import torch.nn as nn
    import fsspec
    from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

    import torch.distributed as dist
    rank = int(os.environ.get("RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    if rank == 0:
        print(f"HF Trainer: world_size={world_size}")

    s3_endpoint = os.environ.get("AWS_ENDPOINT_URL_S3", "")
    storage_options = {"endpoint_url": s3_endpoint} if s3_endpoint else {}
    fs, _ = fsspec.core.url_to_fs(data_dir, **storage_options)

    # ── Rank 0 loads all data from S3, processes it into tensors, then broadcasts ──
    if rank == 0:

        def _load_parquet_dir(path, columns=None, limit=0):
            files = sorted([f for f in fs.ls(path) if f.endswith(".parquet")])
            dfs, rows = [], 0
            for f in files:
                with fs.open(f, "rb") as fh:
                    chunk = pd.read_parquet(io.BytesIO(fh.read()), columns=columns)
                dfs.append(chunk)
                rows += len(chunk)
                if limit > 0 and rows >= limit:
                    break
            return pd.concat(dfs, ignore_index=True)

        df = _load_parquet_dir(
            f"{data_dir}/interactions",
            columns=["item_id", "user_id", "rating", "user_primary_category"],
            limit=max_rows,
        )
        if max_rows > 0:
            df = df.head(max_rows)
        print(f"Loaded {len(df):,} interactions")

        item_meta = _load_parquet_dir(
            f"{data_dir}/offline/item_metadata", columns=["item_id", "item_category"]
        ).drop_duplicates("item_id")
        df = df.merge(item_meta, on="item_id", how="left")

        # ── Super-category mapping (hardcoded — avoids raw S3 scan entirely) ──
        PHOTO_AUDIO = {"Camera & Photo", "Home Audio & Theater", "Portable Audio & Accessories",
                       "GPS & Navigation", "Car Electronics", "Musical Instruments"}
        DIY_OUTDOORS = {"Tools & Home Improvement", "Industrial & Scientific",
                        "Sports & Outdoors", "Automotive", "Arts, Crafts & Sewing", "Office Products"}
        ELECTRONICS = {"Computers", "All Electronics", "Cell Phones & Accessories",
                       "Electronics", "Amazon Devices", "Apple Products", "Amazon Fire TV",
                       "AMAZON FASHION", "Video Games", "Software"}
        BOOKS = {"Books", "Buy a Kindle"}
        PARENT_RENAME = {"Electronics": "Tech & Computing", "Books": "Books & Media",
                         "Home_and_Kitchen": "Home & Living"}

        def to_super_category(main_cat):
            if pd.isna(main_cat) or main_cat is None or str(main_cat) == "None":
                return None
            if main_cat in PHOTO_AUDIO:
                return "Photography & Audio"
            if main_cat in DIY_OUTDOORS:
                return "DIY & Outdoors"
            if main_cat in ELECTRONICS:
                return "Tech & Computing"
            if main_cat in BOOKS:
                return "Books & Media"
            return "Home & Living"

        df["item_category"] = df["item_category"].map(to_super_category)
        null_mask = df["item_category"].isna()
        if null_mask.any():
            df.loc[null_mask, "item_category"] = (
                df.loc[null_mask, "user_primary_category"]
                .map(PARENT_RENAME)
                .fillna("Home & Living")
            )
            print(f"Filled {null_mask.sum():,} null item_categories from user_primary_category")

        counts = df.groupby(["user_id", "item_category"], observed=True).size().reset_index(name="_cnt")
        best_idx = counts.groupby("user_id")["_cnt"].idxmax()
        user_top_cat = counts.loc[best_idx].set_index("user_id")["item_category"]
        del counts, best_idx
        df["user_primary_category"] = df["user_id"].map(user_top_cat).fillna("Home & Living")
        del user_top_cat
        gc.collect()
        print(f"Super-categories: {sorted(df['item_category'].unique())}")

        # ── Encode IDs + categories ──
        user_ids = df["user_id"].astype("category")
        item_ids = df["item_id"].astype("category")
        n_users, n_items = user_ids.cat.categories.size, item_ids.cat.categories.size
        user_to_idx = dict(zip(user_ids.cat.categories.tolist(), range(n_users)))
        item_to_idx = dict(zip(item_ids.cat.categories.tolist(), range(n_items)))

        all_categories = sorted(set(df["item_category"].unique()) | set(df["user_primary_category"].unique()))
        cat_to_idx = {c: i for i, c in enumerate(all_categories)}
        n_categories = len(all_categories)

        user_idx = torch.tensor(user_ids.cat.codes.values, dtype=torch.long)
        item_idx = torch.tensor(item_ids.cat.codes.values, dtype=torch.long)
        user_cat_idx = torch.tensor(df["user_primary_category"].map(cat_to_idx).values, dtype=torch.long)
        item_cat_idx = torch.tensor(df["item_category"].map(cat_to_idx).values, dtype=torch.long)
        labels = torch.tensor((df["rating"].values >= 4).astype(np.float32))

        item_id_to_cat = torch.zeros(n_items, dtype=torch.long)
        for iid, cidx in zip(item_ids.cat.codes.values, item_cat_idx.numpy()):
            item_id_to_cat[iid] = cidx
        user_id_to_cat = torch.zeros(n_users, dtype=torch.long)
        for uid, cidx in zip(user_ids.cat.codes.values, user_cat_idx.numpy()):
            user_id_to_cat[uid] = cidx

        # Free DataFrame before negative sampling
        del df, item_meta
        gc.collect()
        print(f"Freed DataFrames, generating negatives...")

        # Negative sampling (2:1 to reduce memory)
        n_pos = len(labels)
        n_neg = n_pos * 2
        neg_users = torch.randint(0, n_users, (n_neg,))
        neg_items = torch.randint(0, n_items, (n_neg,))
        neg_user_cats = user_id_to_cat[neg_users]
        neg_item_cats = item_id_to_cat[neg_items]

        all_users = torch.cat([user_idx, neg_users]); del user_idx, neg_users
        all_items = torch.cat([item_idx, neg_items]); del item_idx, neg_items
        all_user_cats = torch.cat([user_cat_idx, neg_user_cats]); del user_cat_idx, neg_user_cats
        all_item_cats = torch.cat([item_cat_idx, neg_item_cats]); del item_cat_idx, neg_item_cats
        all_labels = torch.cat([labels, torch.zeros(n_neg)]); del labels
        del user_ids, item_ids
        gc.collect()

        n = len(all_labels)
        perm = torch.randperm(n)
        split = int(0.9 * n)

        print(f"Data ready: {n_users:,} users, {n_items:,} items, {n_categories} cats, "
              f"{n:,} samples (incl. negatives)")

        # Pack metadata for broadcast
        meta_tensor = torch.tensor([n_users, n_items, n_categories, n, split], dtype=torch.long)
    else:
        # Placeholders — will be filled by broadcast
        meta_tensor = torch.zeros(5, dtype=torch.long)
        user_to_idx, item_to_idx, cat_to_idx = {}, {}, {}

    # ── Share processed data: rank 0 saves to S3, other ranks load from S3 ──
    staging_path = f"{data_dir}/training-staging"
    if rank == 0:
        staging_data = {
            "meta": meta_tensor, "all_users": all_users, "all_items": all_items,
            "all_user_cats": all_user_cats, "all_item_cats": all_item_cats,
            "all_labels": all_labels, "perm": perm,
            "item_id_to_cat": item_id_to_cat, "user_id_to_cat": user_id_to_cat,
            "user_to_idx": user_to_idx, "item_to_idx": item_to_idx, "cat_to_idx": cat_to_idx,
        }
        fs.makedirs(staging_path, exist_ok=True)
        staging_file = f"{staging_path}/train_data.pt"
        with fs.open(staging_file, "wb") as f:
            torch.save(staging_data, f)
        fsize = fs.info(staging_file).get("size", 0) / 1e6
        print(f"Rank 0: saved staging data to {staging_file} ({fsize:.1f} MB)")
        del staging_data; gc.collect()
    else:
        # Wait for rank 0 to finish writing
        import time as _wait_time
        for attempt in range(120):
            try:
                if fs.exists(f"{staging_path}/train_data.pt"):
                    break
            except Exception:
                pass
            _wait_time.sleep(2)
        with fs.open(f"{staging_path}/train_data.pt", "rb") as f:
            staging_data = torch.load(io.BytesIO(f.read()), weights_only=False)
        meta_tensor = staging_data["meta"]
        all_users = staging_data["all_users"]
        all_items = staging_data["all_items"]
        all_user_cats = staging_data["all_user_cats"]
        all_item_cats = staging_data["all_item_cats"]
        all_labels = staging_data["all_labels"]
        perm = staging_data["perm"]
        item_id_to_cat = staging_data["item_id_to_cat"]
        user_id_to_cat = staging_data["user_id_to_cat"]
        user_to_idx = staging_data["user_to_idx"]
        item_to_idx = staging_data["item_to_idx"]
        cat_to_idx = staging_data["cat_to_idx"]
        del staging_data
        print(f"Rank {rank}: loaded staging data from S3")

    n_users, n_items, n_categories, n, split = meta_tensor.tolist()
    perm_np = perm.numpy()

    # ── Dataset: returns dicts for HF Trainer ──
    class RecDataset(torch.utils.data.Dataset):
        def __init__(self, users, items, user_cats, item_cats, labels):
            self.users, self.items = users, items
            self.user_cats, self.item_cats = user_cats, item_cats
            self.labels = labels
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, idx):
            return {
                "users": self.users[idx], "items": self.items[idx],
                "user_cats": self.user_cats[idx], "item_cats": self.item_cats[idx],
                "labels": self.labels[idx],
            }

    train_ds = RecDataset(
        all_users[perm_np[:split]], all_items[perm_np[:split]],
        all_user_cats[perm_np[:split]], all_item_cats[perm_np[:split]],
        all_labels[perm_np[:split]],
    )
    val_ds = RecDataset(
        all_users[perm_np[split:]], all_items[perm_np[split:]],
        all_user_cats[perm_np[split:]], all_item_cats[perm_np[split:]],
        all_labels[perm_np[split:]],
    )

    # ── Model: forward returns {"loss", "logits"} for Trainer ──
    class TwoTower(nn.Module):
        def __init__(self, n_users, n_items, n_categories, embed_dim, hidden_dim):
            super().__init__()
            self.user_embed = nn.Embedding(n_users, embed_dim)
            self.item_embed = nn.Embedding(n_items, embed_dim)
            self.cat_embed = nn.Embedding(n_categories, embed_dim // 4)
            cat_dim = embed_dim + embed_dim // 4
            self.user_mlp = nn.Sequential(nn.Linear(cat_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden_dim, embed_dim))
            self.item_mlp = nn.Sequential(nn.Linear(cat_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden_dim, embed_dim))
            self.criterion = nn.BCEWithLogitsLoss()
            self.config = type("Config", (), {
                "keys_to_ignore_at_inference": [],
                "to_dict": lambda self_: {"model_type": "two-tower-cat-rec", "embed_dim": embed_dim, "hidden_dim": hidden_dim, "n_categories": n_categories},
            })()

        def forward(self, users, items, user_cats, item_cats, labels=None):
            u = self.user_mlp(torch.cat([self.user_embed(users), self.cat_embed(user_cats)], dim=-1))
            i = self.item_mlp(torch.cat([self.item_embed(items), self.cat_embed(item_cats)], dim=-1))
            logits = (u * i).sum(dim=1)
            loss = self.criterion(logits, labels) if labels is not None else None
            return {"loss": loss, "logits": logits}

    model = TwoTower(n_users, n_items, n_categories, embed_dim, hidden_dim)
    if rank == 0:
        print(f"Model: n_users={n_users:,}, n_items={n_items:,}, n_categories={n_categories}, "
              f"params={sum(p.numel() for p in model.parameters()):,}")

    # ── Custom collator ──
    def rec_collator(features):
        return {
            "users": torch.stack([f["users"] for f in features]),
            "items": torch.stack([f["items"] for f in features]),
            "user_cats": torch.stack([f["user_cats"] for f in features]),
            "item_cats": torch.stack([f["item_cats"] for f in features]),
            "labels": torch.stack([f["labels"] for f in features]),
        }

    # ── Accuracy metric ──
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = (logits > 0).astype(float)
        return {"accuracy": float((preds == labels).mean())}

    # ── MLflow setup (rank 0 only) ──
    report_to = "none"
    if rank == 0:
        try:
            import mlflow, logging
            logging.getLogger("mlflow.tracing.export.mlflow_v3").setLevel(logging.ERROR)
            mlflow.tracing.disable()
            os.environ.setdefault("MLFLOW_TRACKING_INSECURE_TLS", "true")
            workspace = os.environ.pop("MLFLOW_WORKSPACE", None)
            tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "").rstrip("/")
            if tracking_uri.endswith("/mlflow"):
                tracking_uri = tracking_uri[:-len("/mlflow")]
                os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
            if tracking_uri:
                mlflow.set_tracking_uri(tracking_uri)
            if workspace:
                from mlflow.utils import rest_utils as _ru
                _orig = _ru.http_request
                def _ws(*a, **kw):
                    h = kw.get("extra_headers", {}) or {}
                    h["X-MLflow-Workspace"] = workspace
                    kw["extra_headers"] = h
                    return _orig(*a, **kw)
                _ru.http_request = _ws
            mlflow.set_experiment("smartshop-rec-training")
            report_to = "mlflow"
        except Exception as e:
            print(f"MLflow init failed (non-fatal): {e}")

    # ── TrainingArguments ──
    training_args = TrainingArguments(
        output_dir="/tmp/rec-checkpoints",
        run_name=f"twotower-{world_size}gpu",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=1e-5,
        max_grad_norm=1.0,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to=report_to,
        bf16=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        ddp_find_unused_parameters=False,
    )

    hf_trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=rec_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    train_start = _time.time()
    train_result = hf_trainer.train()
    total_time = _time.time() - train_start
    metrics = train_result.metrics

    # Save model BEFORE eval — ensures checkpoint persists even if eval crashes
    best_state = hf_trainer.model.state_dict()

    if rank == 0:
        checkpoint = {
            "model_state_dict": best_state,
            "n_users": n_users, "n_items": n_items,
            "n_categories": n_categories,
            "embed_dim": embed_dim, "hidden_dim": hidden_dim,
            "user_to_idx": user_to_idx, "item_to_idx": item_to_idx,
            "cat_to_idx": cat_to_idx,
            "item_id_to_cat": item_id_to_cat.tolist(),
            "user_id_to_cat": user_id_to_cat.tolist(),
            "epoch": int(metrics.get("epoch", epochs)),
            "train_loss": metrics.get("train_loss", 0),
        }
        buf = io.BytesIO()
        torch.save(checkpoint, buf)
        buf.seek(0)
        out_fs, _ = fsspec.core.url_to_fs(output_dir, **storage_options)
        out_fs.makedirs(output_dir, exist_ok=True)
        with out_fs.open(f"{output_dir}/best_model.pt", "wb") as f:
            f.write(buf.read())
        print(f"\nModel saved to: {output_dir}/best_model.pt")

    # Eval after save — failure here won't lose the model
    eval_metrics = {}
    try:
        eval_metrics = hf_trainer.evaluate()
    except Exception as e:
        if rank == 0:
            print(f"Evaluation failed (model already saved): {e}")

    if rank == 0:
        print(f"Training complete in {total_time:.0f}s")
        print(f"  train_loss={metrics.get('train_loss', 0):.4f}")
        if eval_metrics:
            print(f"  eval_loss={eval_metrics.get('eval_loss', 0):.4f}")
            print(f"  eval_accuracy={eval_metrics.get('eval_accuracy', 0):.4f}")

        # Log to MLflow
        if report_to == "mlflow":
            try:
                mlflow.log_params({
                    "architecture": "two-tower",
                    "n_users": n_users,
                    "n_items": n_items,
                    "embed_dim": embed_dim,
                    "hidden_dim": hidden_dim,
                    "neg_sampling_ratio": 3,
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "world_size": world_size,
                })
                log_m = {
                    "final_train_loss": metrics.get("train_loss", 0),
                    "training_time_s": total_time,
                    "stopped_epoch": int(metrics.get("epoch", epochs)),
                }
                if eval_metrics:
                    log_m["best_eval_loss"] = eval_metrics.get("eval_loss", 0)
                    log_m["best_eval_accuracy"] = eval_metrics.get("eval_accuracy", 0)
                mlflow.log_metrics(log_m)
                mlflow.pytorch.log_model(
                    hf_trainer.model,
                    artifact_path="model",
                    registered_model_name="smartshop-rec-twotower",
                )
                print("Model logged to MLflow ✓")
            except Exception as e:
                print(f"MLflow model logging failed (non-fatal): {e}")

print("train_fn defined ✓")

train_fn defined ✓


## 1.3 Submit Rec TrainJob


In [48]:
import base64
from datetime import datetime
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig
from kubeflow.trainer.options import (
    Name, Labels, PodTemplateOverrides, PodTemplateOverride,
    PodSpecOverride, ContainerOverride
)

v1 = k8s.CoreV1Api()
mlflow_secret = v1.read_namespaced_secret(MLFLOW_SECRET, NAMESPACE)

def decode(secret, key):
    return base64.b64decode(secret.data[key]).decode()

MLFLOW_TRACKING_URI = decode(mlflow_secret, "MLFLOW_TRACKING_URI")
print(f"MLflow URI (from secret): {MLFLOW_TRACKING_URI}")

REC_JOB_NAME = f"rec-train-{datetime.now().strftime('%m%d-%H%M')}"

job = trainer.train(
    trainer=TransformersTrainer(
        func=train_fn,
        func_args={
            "data_dir": rec_params["data_dir"],
            "output_dir": rec_params["output_dir"],
            "epochs": rec_params["epochs"],
            "batch_size": rec_params["batch_size"],
            "lr": rec_params["lr"],
            "embed_dim": rec_params["embed_dim"],
            "hidden_dim": rec_params["hidden_dim"],
            "max_rows": rec_params["max_rows"],
        },
        num_nodes=rec_params["num_nodes"],
        resources_per_node={"gpu": rec_params["gpus_per_node"], "cpu": rec_params["cpu_per_node"], "memory": rec_params["memory_per_node"]},
        env={
            "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
            "MLFLOW_TRACKING_INSECURE_TLS": "true",
            "MLFLOW_TRACKING_TOKEN": decode(mlflow_secret, "MLFLOW_TRACKING_TOKEN"),
            "MLFLOW_WORKSPACE": decode(mlflow_secret, "MLFLOW_WORKSPACE"),
            "AWS_ENDPOINT_URL_S3": MINIO_ENDPOINT,
            "NCCL_DEBUG": "INFO",
            "NCCL_IB_DISABLE": "1",
        },
        enable_progression_tracking=True,
        output_dir=f"s3://{S3_MODELS_BUCKET}/rec-checkpoints",
        data_connection_name="smartshop-s3-credentials",
        periodic_checkpoint_config=PeriodicCheckpointConfig(
            save_strategy="epoch",
            save_total_limit=2,
        ),
        verify_cloud_storage_ssl=False,
    ),
    runtime=runtime,
    options=[
        Name(REC_JOB_NAME),
        Labels({"app": "smartshop", "component": "rec-training",
                "kueue.x-k8s.io/queue-name": "smartshop-training"}),
        PodTemplateOverrides(PodTemplateOverride(
            target_jobs=["node"],
            spec=PodSpecOverride(
                volumes=[{"name": "shm", "emptyDir": {"medium": "Memory"}}],
                containers=[ContainerOverride(
                    name="node",
                    volume_mounts=[{"name": "shm", "mountPath": "/dev/shm"}],
                )]
            )
        ))
    ]
)

print(f"Submitted TrainJob: {REC_JOB_NAME}")
print(f"Topology: {rec_params['num_nodes']} nodes x {rec_params['gpus_per_node']} GPUs = {rec_params['num_nodes'] * rec_params['gpus_per_node']} total GPUs")


MLflow URI (from secret): https://mlflow.redhat-ods-applications.svc.cluster.local:8443
Submitted TrainJob: rec-train-0505-0642
Topology: 4 nodes x 2 GPUs = 8 total GPUs


## 1.4 Monitor Rec Training


In [ ]:
print(f"Waiting for {REC_JOB_NAME} to start running...")
trainer.wait_for_job_status(name=REC_JOB_NAME, status={"Running"}, timeout=600)
print(f"{REC_JOB_NAME} is Running ✓")

print(f"\nWaiting for completion (timeout {TIMEOUT_SECONDS}s)...")
trainer.wait_for_job_status(name=REC_JOB_NAME, status={"Complete", "Failed"}, timeout=TIMEOUT_SECONDS)

status = trainer.get_job(REC_JOB_NAME)
print(f"\nFinal status: {status.status}")


## 1.5 Verify Model Artifact


In [49]:
import fsspec

REC_OUTPUT_DIR = rec_params["output_dir"]
fs, _ = fsspec.core.url_to_fs(REC_OUTPUT_DIR, endpoint_url=MINIO_ENDPOINT)
files = fs.ls(REC_OUTPUT_DIR)

print(f"Model artifacts in {REC_OUTPUT_DIR}:")
for f in files:
    info = fs.info(f)
    size_mb = info.get("size", 0) / (1024 * 1024)
    print(f"  {os.path.basename(f):40s} {size_mb:>8.1f} MB")

assert any("best_model.pt" in f for f in files), "best_model.pt not found!"
print("\nbest_model.pt verified ✓")


Model artifacts in s3://smartshop-models/recommendation:
  best_model.pt                               773.5 MB

best_model.pt verified ✓


---
# Phase 2: LLM Fine-Tuning (LoRA + FSDP)

## 2.1 LLM Training Parameters

Edit these to customize LLM fine-tuning.


In [ ]:
%%yaml llm_params

# Model
base_model: mistralai/Mistral-7B-Instruct-v0.3

# Data — bucket names must match .env S3_FEATURES_BUCKET / S3_MODELS_BUCKET
data_dir: s3://smartshop-features/llm_data
output_dir: s3://smartshop-models/llm-adapter

# Hyperparameters
epochs: 1
batch_size: 4
lr: 0.0002
max_steps: 1500
gradient_accumulation: 2
max_seq_length: 2048
max_files: 3                               # data files to load

# LoRA
lora_r: 16
lora_alpha: 32

# Cluster topology
num_nodes: 4
gpus_per_node: 2
cpu_per_node: 8
memory_per_node: 128Gi


In [ ]:
print(f"Base model:   {llm_params['base_model']}")
print(f"Data:         {llm_params['data_dir']}")
print(f"Output:       {llm_params['output_dir']}")
print(f"LoRA:         r={llm_params['lora_r']}, alpha={llm_params['lora_alpha']}")
print(f"Topology:     {llm_params['num_nodes']} nodes x {llm_params['gpus_per_node']} GPUs = {llm_params['num_nodes'] * llm_params['gpus_per_node']} total GPUs")
print(f"FSDP:         full_shard + activation_checkpointing")


## 2.2 Define Training Function

Self-contained LoRA + FSDP fine-tuning function.
Uses `transformers` SFTTrainer with FSDP `full_shard` (ZeRO-3 equivalent).


In [51]:
def train_llm(
    data_dir: str,
    output_dir: str,
    base_model: str,
    epochs: int = 1,
    max_steps: int = -1,
    batch_size: int = 4,
    gradient_accumulation: int = 4,
    lr: float = 2e-4,
    lora_r: int = 16,
    lora_alpha: int = 32,
    max_seq_length: int = 2048,
    max_files: int = 3,
):
    import os, tempfile, time as _time
    import fsspec
    import torch
    from datasets import load_dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTConfig, SFTTrainer

    rank = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    if rank == 0:
        print(f"Fine-tuning {base_model} with LoRA + FSDP")
        print(f"World size: {world_size}")

    # Load data from S3
    s3_endpoint = os.environ.get("AWS_ENDPOINT_URL_S3", "")
    storage_opts = {}
    if data_dir.startswith("s3://") and s3_endpoint:
        storage_opts = {
            "key": os.environ.get("AWS_ACCESS_KEY_ID", ""),
            "secret": os.environ.get("AWS_SECRET_ACCESS_KEY", ""),
            "client_kwargs": {"endpoint_url": s3_endpoint},
        }

    def _load_split(split):
        split_path = data_dir.rstrip("/") + "/" + split
        fs, _ = fsspec.core.url_to_fs(split_path, **({
            "endpoint_url": s3_endpoint} if s3_endpoint else {}))
        if not fs.exists(split_path):
            return None
        entries = sorted(fs.ls(split_path, detail=False))
        parts = [e for e in entries if any(e.endswith(ext) for ext in [".jsonl", ".txt", ".parquet"]) or "part-" in os.path.basename(e)]
        if max_files > 0:
            parts = parts[:max_files]
        if data_dir.startswith("s3://") and parts and not parts[0].startswith("s3://"):
            parts = [f"s3://{f}" for f in parts]
        if rank == 0:
            print(f"  {split}: {len(parts)} files")
        return load_dataset("json", data_files=parts, split="train", storage_options=storage_opts or None)

    train_dataset = _load_split("train")
    val_dataset = _load_split("val")
    if train_dataset is None:
        raise RuntimeError(f"No training data at {data_dir}/train")
    if rank == 0:
        print(f"Train: {len(train_dataset):,} examples")

    def format_instruction(example):
        instruction = example.get("instruction", "")
        input_text = example.get("input", "")
        output_text = example.get("output", "")
        prompt = f"[INST] {instruction}\n\n{input_text} [/INST]" if input_text else f"[INST] {instruction} [/INST]"
        return f"{prompt} {output_text}" if output_text else prompt

    # Model download sync (local_rank 0 fetches, others wait)
    _lock = os.path.join(os.environ.get("HF_HOME", "/tmp/hf_home"), ".model_ready")
    if local_rank == 0:
        from huggingface_hub import snapshot_download
        snapshot_download(base_model)
        open(_lock, "w").close()
    else:
        import time as _tw
        while not os.path.exists(_lock):
            _tw.sleep(2)

    model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.bfloat16, trust_remote_code=True)
    peft_config = LoraConfig(
        r=lora_r, lora_alpha=lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    if rank == 0:
        model.print_trainable_parameters()

    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ── MLflow setup (rank 0 only) ──
    report_to = "none"
    if rank == 0:
        try:
            import mlflow, logging
            logging.getLogger("mlflow.tracing.export.mlflow_v3").setLevel(logging.ERROR)
            mlflow.tracing.disable()
            os.environ.setdefault("MLFLOW_TRACKING_INSECURE_TLS", "true")
            workspace = os.environ.pop("MLFLOW_WORKSPACE", None)
            tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "").rstrip("/")
            if tracking_uri.endswith("/mlflow"):
                tracking_uri = tracking_uri[:-len("/mlflow")]
                os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
            if tracking_uri:
                mlflow.set_tracking_uri(tracking_uri)
            if workspace:
                from mlflow.utils import rest_utils as _ru
                _orig = _ru.http_request
                def _ws(*a, **kw):
                    h = kw.get("extra_headers", {}) or {}
                    h["X-MLflow-Workspace"] = workspace
                    kw["extra_headers"] = h
                    return _orig(*a, **kw)
                _ru.http_request = _ws
            mlflow.set_experiment("smartshop-llm-finetuning")
            report_to = "mlflow"
        except Exception as e:
            print(f"MLflow init failed (non-fatal): {e}")

    # SFT training with FSDP
    training_args = SFTConfig(
        output_dir="/tmp/llm-checkpoints",
        run_name=f"lora-fsdp-{world_size}gpu",
        num_train_epochs=epochs, max_steps=max_steps,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        learning_rate=lr, weight_decay=0.01,
        warmup_ratio=0.1, lr_scheduler_type="cosine",
        logging_steps=10, save_strategy="steps", save_steps=300, save_total_limit=2,
        bf16=True, report_to=report_to, max_length=max_seq_length,
        fsdp="full_shard auto_wrap",
        fsdp_config={"transformer_layer_cls_to_wrap": ["MistralDecoderLayer"]},
    )

    hf_trainer = SFTTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=val_dataset,
        processing_class=tokenizer, formatting_func=format_instruction,
    )

    train_start = _time.time()
    train_result = hf_trainer.train()
    total_time = _time.time() - train_start
    metrics = train_result.metrics
    train_loss = metrics.get("train_loss", 0)

    # Save model BEFORE eval — ensures adapter persists even if eval crashes
    is_s3 = output_dir.startswith("s3://")
    local_dir = tempfile.mkdtemp(prefix="llm-adapter-") if is_s3 else output_dir
    hf_trainer.save_model(local_dir)
    tokenizer.save_pretrained(local_dir)

    if rank == 0 and is_s3:
        fs, _ = fsspec.core.url_to_fs(output_dir, **({
            "endpoint_url": s3_endpoint} if s3_endpoint else {}))
        fs.put(local_dir, output_dir, recursive=True)
        print(f"LoRA adapter uploaded to {output_dir}")

    # Eval after save — failure here won't lose the adapter
    eval_metrics = {}
    if val_dataset is not None:
        try:
            eval_metrics = hf_trainer.evaluate()
        except Exception as e:
            if rank == 0:
                print(f"Evaluation failed (adapter already saved): {e}")

    if rank == 0:
        print(f"Training complete in {total_time:.0f}s, train_loss={train_loss:.4f}")
        if eval_metrics:
            print(f"  eval_loss={eval_metrics.get('eval_loss', 0):.4f}")

        # Log to MLflow
        if report_to == "mlflow":
            try:
                mlflow.log_params({
                    "base_model": base_model,
                    "lora_r": lora_r,
                    "lora_alpha": lora_alpha,
                    "lora_target_modules": "q,k,v,o,gate,up,down",
                    "fsdp_strategy": "full_shard",
                    "max_seq_length": max_seq_length,
                    "train_examples": len(train_dataset),
                    "val_examples": len(val_dataset) if val_dataset else 0,
                    "world_size": world_size,
                    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
                    "total_params": sum(p.numel() for p in model.parameters()),
                })
                log_metrics = {
                    "final_train_loss": train_loss,
                    "training_time_s": total_time,
                    "tokens_per_sec": metrics.get("train_samples_per_second", 0) * batch_size * max_seq_length,
                }
                if eval_metrics:
                    log_metrics["eval_loss"] = eval_metrics.get("eval_loss", 0)
                mlflow.log_metrics(log_metrics)
                mlflow.log_artifacts(local_dir, artifact_path="lora-adapter")
                print("Model logged to MLflow ✓")
            except Exception as e:
                print(f"MLflow model logging failed (non-fatal): {e}")

print("train_llm defined ✓")

train_llm defined ✓


## 2.3 Submit LLM TrainJob


In [52]:
hf_secret = v1.read_namespaced_secret(HF_SECRET, NAMESPACE)

LLM_JOB_NAME = f"llm-finetune-{datetime.now().strftime('%m%d-%H%M')}"

job = trainer.train(
    trainer=TransformersTrainer(
        func=train_llm,
        func_args={
            "data_dir": llm_params["data_dir"],
            "output_dir": llm_params["output_dir"],
            "base_model": llm_params["base_model"],
            "epochs": llm_params["epochs"],
            "max_steps": llm_params["max_steps"],
            "batch_size": llm_params["batch_size"],
            "gradient_accumulation": llm_params["gradient_accumulation"],
            "lr": llm_params["lr"],
            "lora_r": llm_params["lora_r"],
            "lora_alpha": llm_params["lora_alpha"],
            "max_seq_length": llm_params["max_seq_length"],
            "max_files": llm_params["max_files"],
        },
        num_nodes=llm_params["num_nodes"],
        resources_per_node={"gpu": llm_params["gpus_per_node"], "cpu": llm_params["cpu_per_node"], "memory": llm_params["memory_per_node"]},
        env={
            "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
            "MLFLOW_TRACKING_INSECURE_TLS": "true",
            "MLFLOW_TRACKING_TOKEN": decode(mlflow_secret, "MLFLOW_TRACKING_TOKEN"),
            "MLFLOW_WORKSPACE": decode(mlflow_secret, "MLFLOW_WORKSPACE"),
            "AWS_ENDPOINT_URL_S3": MINIO_ENDPOINT,
            "HF_TOKEN": decode(hf_secret, "token"),
            "HF_HOME": "/tmp/hf_home",
            "NCCL_DEBUG": "INFO",
            "NCCL_IB_DISABLE": "1",
        },
        enable_progression_tracking=True,
        output_dir=f"s3://{S3_MODELS_BUCKET}/llm-checkpoints",
        data_connection_name="smartshop-s3-credentials",
        periodic_checkpoint_config=PeriodicCheckpointConfig(
            save_strategy="steps",
            save_steps=300,
            save_total_limit=2,
        ),
        verify_cloud_storage_ssl=False,
    ),
    runtime=runtime,
    options=[
        Name(LLM_JOB_NAME),
        Labels({"app": "smartshop", "component": "llm-training",
                "kueue.x-k8s.io/queue-name": "smartshop-training"}),
        PodTemplateOverrides(PodTemplateOverride(
            target_jobs=["node"],
            spec=PodSpecOverride(
                volumes=[{"name": "shm", "emptyDir": {"medium": "Memory"}}],
                containers=[ContainerOverride(
                    name="node",
                    volume_mounts=[{"name": "shm", "mountPath": "/dev/shm"}],
                )]
            )
        ))
    ]
)

print(f"Submitted TrainJob: {LLM_JOB_NAME}")
print(f"Base model: {llm_params['base_model']}")
print(f"LoRA r={llm_params['lora_r']}, alpha={llm_params['lora_alpha']}, FSDP full_shard")
print(f"Topology: {llm_params['num_nodes']} nodes x {llm_params['gpus_per_node']} GPUs")


Submitted TrainJob: llm-finetune-0505-0652
Base model: mistralai/Mistral-7B-Instruct-v0.3
LoRA r=16, alpha=32, FSDP full_shard
Topology: 4 nodes x 2 GPUs


## 2.4 Monitor LLM Training


In [ ]:
print(f"Waiting for {LLM_JOB_NAME} to start running...")
trainer.wait_for_job_status(name=LLM_JOB_NAME, status={"Running"}, timeout=600)
print(f"{LLM_JOB_NAME} is Running ✓")

print(f"\nWaiting for completion (timeout {TIMEOUT_SECONDS}s)...")
trainer.wait_for_job_status(name=LLM_JOB_NAME, status={"Complete", "Failed"}, timeout=TIMEOUT_SECONDS)

status = trainer.get_job(LLM_JOB_NAME)
print(f"\nFinal status: {status.status}")


## 2.5 Verify Adapter Artifact


In [ ]:
LLM_OUTPUT_DIR = llm_params["output_dir"]
fs, _ = fsspec.core.url_to_fs(LLM_OUTPUT_DIR, endpoint_url=MINIO_ENDPOINT)
files = fs.ls(LLM_OUTPUT_DIR)

print(f"Adapter artifacts in {LLM_OUTPUT_DIR}:")
for f in files:
    info = fs.info(f)
    size_mb = info.get("size", 0) / (1024 * 1024)
    print(f"  {os.path.basename(f):40s} {size_mb:>8.1f} MB")

assert any("adapter_config.json" in f for f in files), "adapter_config.json not found!"
print("\nLoRA adapter verified ✓")


Adapter artifacts in s3://smartshop-models/llm-adapter:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   160.1 MB
  chat_template.jinja                           0.0 MB
  optimizer.bin                               320.4 MB
  pytorch_model_fsdp.bin                      160.2 MB
  rng_state_0.pth                               0.0 MB
  rng_state_1.pth                               0.0 MB
  rng_state_2.pth                               0.0 MB
  rng_state_3.pth                               0.0 MB
  rng_state_4.pth                               0.0 MB
  rng_state_5.pth                               0.0 MB
  rng_state_6.pth                               0.0 MB
  rng_state_7.pth                               0.0 MB
  scheduler.pt                                  0.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                                3.5 MB
  tokeniz

---
## Summary

| Phase | Model | Architecture | Output |
|-------|-------|-------------|--------|
| **1** | Recommendation | Two-Tower HF Trainer + DDP | `best_model.pt` |
| **2** | LLM | Mistral-7B + LoRA + FSDP | `llm-adapter/` |

**Key RHOAI capabilities shown:**
- Kubeflow Trainer SDK for distributed training orchestration
- HuggingFace Trainer with automatic progress tracking, checkpointing, and early stopping
- Multi-node DDP with `torch-distributed` ClusterTrainingRuntime
- LoRA + FSDP for memory-efficient LLM fine-tuning
- MLflow experiment tracking (RHOAI-managed)
- S3 model artifact persistence

**Next:** Run `03_embeddings.ipynb` to generate review embeddings for RAG, then `04_serving.ipynb` to deploy all models.
